In [4]:
!pip install unsloth datasets trl accelerate bitsandbytes tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 41.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 39.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 117.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 65.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 MB 31.3 MB/s  0:00:08
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 24.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 19.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 22.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 22.0 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled t

In [5]:
%pip install -q -U modal

In [6]:
import os
import modal
from google.colab import userdata

# Nếu Secrets của bạn đặt tên khác, sửa hai tên trong userdata.get().
token_id = userdata.get("MODAL_TOKEN_ID").strip()
token_secret = userdata.get("MODAL_TOKEN_SECRET").strip()

for name, value in [
    ("MODAL_TOKEN_ID", token_id),
    ("MODAL_TOKEN_SECRET", token_secret),
]:
    assert (
        value
        and value.isascii()
        and not any(c.isspace() for c in value)
    ), f"Giá trị {name} trong Colab Secrets không đúng định dạng token."

# Giữ để cell khởi tạo H200 phía sau sử dụng.
os.environ["MODAL_TOKEN_ID"] = token_id
os.environ["MODAL_TOKEN_SECRET"] = token_secret

# Tạo client MỚI từ đúng giá trị vừa đọc.
modal_client = modal.Client.from_credentials(
    token_id,
    token_secret,
)

app = modal.App("vihsd-h200-colab")

print("✅ Kết nối Modal thành công bằng client mới.")

✅ Kết nối Modal thành công bằng client mới.


In [7]:
import sys

# Dừng Sandbox Jupyter cũ nếu đã tạo, tránh giữ thêm GPU.
if "sandbox" in globals():
    sandbox.terminate()
    del sandbox
    print("Đã dừng Sandbox Jupyter cũ.")

# Hàm định nghĩa trong notebook cần Python phía Modal
# khớp phiên bản Python của Colab.
python_version = f"{sys.version_info.major}.{sys.version_info.minor}"

volume = modal.Volume.from_name(
    "qwen3-vihsd-data",
    create_if_missing=True,
    version=2,
)

image = (
    modal.Image.from_registry(
        "nvidia/cuda:12.8.1-devel-ubuntu22.04",
        add_python=python_version,
    )
    .apt_install("git", "build-essential")
    .pip_install(
        "unsloth",
        "datasets",
        "trl",
        "accelerate",
        "bitsandbytes",
        "tqdm",
        "scikit-learn",
        "wandb",
    )
    .env({
        "HF_HOME": "/data/hf_cache",
        "WANDB_MODE": "offline",
    })
)

print(f"✅ Đã khai báo môi trường Modal: Python {python_version}")
print("Chưa cấp GPU ở cell này. Cell tiếp theo sẽ gọi H200.")

✅ Đã khai báo môi trường Modal: Python 3.13
Chưa cấp GPU ở cell này. Cell tiếp theo sẽ gọi H200.


In [8]:
@app.function(
    image=image,
    gpu="H200",
    cpu=4,
    memory=32768,
    volumes={"/data": volume},
    timeout=600,
)
def check_modal_gpu():
    # Các import này chạy trên Modal, không chạy trên Colab.
    import unsloth
    import torch

    assert torch.cuda.is_available(), "Máy Modal chưa có CUDA."

    gpu_name = torch.cuda.get_device_name(0)
    vram_gib = (
        torch.cuda.get_device_properties(0).total_memory / 1024**3
    )

    assert "H200" in gpu_name, f"GPU được cấp không phải H200: {gpu_name}"

    return {
        "gpu": gpu_name,
        "vram_gib": round(vram_gib, 1),
        "data_dir": "/data",
    }


# Gọi hàm trên H200, nhận kết quả ngay tại Colab.
with modal.enable_output():
    with app.run(client=modal_client):
        gpu_info = check_modal_gpu.remote()

print("\n✅ Kiểm tra GPU phía Modal thành công")
print("GPU:", gpu_info["gpu"])
print("VRAM:", gpu_info["vram_gib"], "GiB")
print("Thư mục trên Modal:", gpu_info["data_dir"])

✓ Initialized. View run at 
https://modal.com/apps/luluchi19/main/ap-y4WPp13a7sGimmkRP6ZJzV
✓ Created objects.
└── ✓ Created Function check_modal_gpu.

== CUDA ==

CUDA Version 12.8.1

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1030 16:06:11.347000 2 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a futu

# Infer sử dụng model gốc "unsloth/Qwen3-4B-Instruct-2507" để làm baseline

In [9]:
import json

# ---------- CONFIG ----------
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"

USER_TEMPLATE = """Bạn là hệ thống kiểm duyệt nội dung tiếng Việt.

Định nghĩa nhãn:
1. CLEAN
Câu không chứa ngôn từ quấy rối, không có từ ngữ thô tục, và không tấn công bất kỳ ai.
2. OFFENSIVE
Câu có chứa nội dung quấy rối hoặc từ ngữ thô tục, nhưng không nhắm vào một đối tượng cụ thể nào, để than phiền chung chung.
3. HATE
Câu chứa nội dung quấy rối, lăng mạ và nhắm trực tiếp vào một cá nhân hoặc một nhóm người cụ thể dựa trên các đặc điểm của họ (như đặc điểm cá nhân, tôn giáo, quốc tịch, chủng tộc, vùng miền...).
Lưu ý: Nếu có chửi thề nhưng không nhắm vào ai cụ thể, phải chọn OFFENSIVE. Chỉ chọn HATE khi có mục tiêu bị nhắm đến cụ thể.

Nhiệm vụ: phân loại câu dưới vào MỘT trong các nhãn CLEAN | OFFENSIVE | HATE
Câu:
\"\"\"{text}\"\"\"

Chỉ trả về JSON hợp lệ với schema:
{{
  "explanation": "Một lời giải thích ngắn gọn lý do phân loại như vậy (sử dụng tiếng Việt)",
  "evidence": ["Trích nguyên văn những từ ngữ, đoạn vi phạm (nếu có)"],
  "label": "CLEAN | OFFENSIVE | HATE"
}}

KHÔNG được trả lời thêm bất kỳ text nào ngoài JSON.
"""


# ---------- CHẠY TRÊN MODAL ----------
@app.function(
    image=image,
    gpu="H200",
    cpu=4,
    memory=32768,
    volumes={"/data": volume},
    timeout=1800,
)
def infer_baseline_modal(user_input, user_template, model_name):
    # Import bên trong hàm để chạy trên máy Modal.
    from unsloth import FastLanguageModel
    import torch

    DEVICE = "cuda"
    assert torch.cuda.is_available(), "Máy Modal chưa có CUDA."
    print("GPU phía Modal:", torch.cuda.get_device_name(0))

    # ---------- LOAD MODEL ----------
    print(f"Loading model: {model_name}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=1024,
        dtype=torch.float16,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.eos_token

    prompt = tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": user_template.format(text=user_input),
            }
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    print("\n\n\n===== PROMPT SENT TO MODEL =====")
    print(prompt)

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # ---------- INFERENCE ----------
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.7,
        )

    gen_ids = outputs[:, inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

    # Trả text về Colab.
    return decoded


# ---------- INPUT TẠI COLAB ----------
user_input = input("\n\nNhập câu cần phân loại: ")

# ---------- GỌI H200 ----------
with modal.enable_output():
    with app.run(client=modal_client):
        decoded = infer_baseline_modal.remote(
            user_input,
            USER_TEMPLATE,
            MODEL_NAME,
        )

# ---------- OUTPUT JSON TẠI COLAB ----------
try:
    obj = json.loads(decoded)
    print("\n\n\n===== PARSED JSON =====")
    print(json.dumps(obj, indent=2, ensure_ascii=False))
except Exception as e:
    print("\nJSON parse failed:", e)
    print("Raw output:", decoded)



Nhập câu cần phân loại: A kể chuyện n lại hay vl 😂
✓ Initialized. View run at 
https://modal.com/apps/luluchi19/main/ap-7wfcbzecuJofw0D1vNW3p7
✓ Created objects.
├── ✓ Created Function check_modal_gpu.
└── ✓ Created Function infer_baseline_modal.

== CUDA ==

CUDA Version 12.8.1

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1030 16:07:27.464000 2 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value

In [10]:
# ---------- CONFIG ----------
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"
BATCH_SIZE = 32

# Giữ nguyên template riêng của cell đánh giá trong bản gốc.
USER_TEMPLATE = """Bạn là hệ thống kiểm duyệt nội dung tiếng Việt.

Định nghĩa nhãn:
1. CLEAN
Câu không chứa ngôn từ quấy rối, không có từ ngữ thô tục, và không tấn công bất kỳ ai.
2. OFFENSIVE
Câu có chứa nội dung quấy rối hoặc từ ngữ thô tục, nhưng không nhắm vào một đối tượng cụ thể nào, để than phiền chung chung.
3. HATE
Câu chứa nội dung quấy rối, lăng mạ và nhắm trực tiếp vào một cá nhân hoặc một nhóm người cụ thể dựa trên các đặc điểm của họ (như đặc điểm cá nhân, tôn giáo, quốc tịch, chủng tộc, vùng miền...).
Lưu ý: Nếu có chửi thề nhưng không nhắm vào ai cụ thể, phải chọn OFFENSIVE. Chỉ chọn HATE khi có mục tiêu bị nhắm đến cụ thể.

Nhiệm vụ: phân loại câu dưới vào MỘT trong các nhãn CLEAN | OFFENSIVE | HATE
Câu:
\"\"\"{text}\"\"\"

Chỉ trả về JSON hợp lệ với schema:
{{
  "explanation": "Một lời giải thích ngắn gọn lý do phân loại như vậy (sử dụng tiếng Việt)",
  "evidence": ["Trích nguyên văn những từ ngữ, đoạn vi phạm (nếu có)"],
  "label": "CLEAN | OFFENSIVE | HATE"
}}"""


# ---------- CHẠY TRÊN MODAL ----------
@app.function(
    image=image,
    gpu="H200",
    cpu=4,
    memory=32768,
    volumes={"/data": volume},
    timeout=6 * 60 * 60,
)
def evaluate_baseline_modal(user_template, model_name, batch_size):
    from unsloth import FastLanguageModel
    import json
    import torch
    from datasets import load_dataset
    from sklearn.metrics import f1_score
    from tqdm import tqdm

    DEVICE = "cuda"
    assert torch.cuda.is_available(), "Máy Modal chưa có CUDA."
    print("GPU phía Modal:", torch.cuda.get_device_name(0))

    # ---------- PARSE (NO FALLBACK) ----------
    def extract_label(text):
        obj = json.loads(text)
        return obj["label"].strip().upper()

    # ---------- LOAD MODEL ----------
    print(f"Loading model: {model_name}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=1024,
        dtype=torch.float16,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.eos_token

    # ---------- LOAD DATA ----------
    ds = load_dataset("vominhmanh/vihsd-explainable")["test"]

    y_true, y_pred = [], []

    print(
        f"Running inference on {len(ds)} samples "
        f"(BS={batch_size})..."
    )

    # ---------- INFERENCE ----------
    for i in tqdm(
        range(0, len(ds), batch_size),
        mininterval=60,
        miniters=1,
    ):
        batch = ds[i:i + batch_size]

        texts = batch["text"]
        labels = batch["label"] if "label" in batch else [
            ["CLEAN", "OFFENSIVE", "HATE"][lid]
            for lid in batch["label_id"]
        ]

        prompts = [
            tokenizer.apply_chat_template(
                [
                    {
                        "role": "user",
                        "content": user_template.format(text=t),
                    }
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            for t in texts
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.7,
            )

        gen_ids = outputs[:, inputs["input_ids"].shape[1]:]
        decoded = tokenizer.batch_decode(
            gen_ids,
            skip_special_tokens=True,
        )

        for t, raw, gold in zip(texts, decoded, labels):
            try:
                pred = extract_label(raw)
            except:
                pred = "ERROR"

            y_true.append(gold)
            y_pred.append(pred)

    # ---------- F1: GIỮ CÁCH TÍNH GỐC ----------
    valid = [
        (t, p)
        for t, p in zip(y_true, y_pred)
        if t is not None
    ]
    y_t, y_p = zip(*valid)

    f1 = f1_score(y_t, y_p, average="macro")

    # Trả kết quả và các nhãn về Colab.
    return {
        "samples": len(y_t),
        "f1": float(f1),
        "y_true": y_true,
        "y_pred": y_pred,
    }


# ---------- GỌI H200 TỪ COLAB ----------
with modal.enable_output():
    with app.run(client=modal_client):
        print("Đang đánh giá baseline. Theo dõi tại:", app.get_dashboard_url())

        baseline_result = evaluate_baseline_modal.remote(
            USER_TEMPLATE,
            MODEL_NAME,
            BATCH_SIZE,
        )

# ---------- KẾT QUẢ TẠI COLAB ----------
y_true = baseline_result["y_true"]
y_pred = baseline_result["y_pred"]
f1 = baseline_result["f1"]

print("\n===== RESULT =====")
print(f"Samples: {baseline_result['samples']}")
print(f"Macro F1: {f1:.4f}")

✓ Initialized. View run at 
https://modal.com/apps/luluchi19/main/ap-JDPSwcJHv9AJGYkf5q65AZ
✓ Created objects.
├── ✓ Created Function check_modal_gpu.
├── ✓ Created Function infer_baseline_modal.
└── ✓ Created Function evaluate_baseline_modal.
Đang đánh giá baseline. Theo dõi tại: https://modal.com/id/ap-JDPSwcJHv9AJGYkf5q65AZ

== CUDA ==

CUDA Version 12.8.1

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1030 16:08:20.202000 2 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an

[modal-client] 2026-10-30T16:28:52+0000 Timed out waiting for final app logs.


✓ App completed. View run at 
https://modal.com/apps/luluchi19/main/ap-JDPSwcJHv9AJGYkf5q65AZ

===== RESULT =====
Samples: 6680
Macro F1: 0.3721


# Training

In [11]:
from google.colab import userdata

wandb_secret = modal.Secret.from_dict({
    "WANDB_API_KEY": userdata.get("WANDB_API_KEY").strip(),
})

In [12]:
def load_base_model():
    from unsloth import FastLanguageModel
    import torch

    assert torch.cuda.is_available(), "Hàm này cần chạy bên trong job Modal."
    print("GPU đang sử dụng:", torch.cuda.get_device_name(0))

    # -----------------------------
    # 3. LOAD MODEL
    # -----------------------------
    base_model, base_tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen3-4B-Instruct-2507",
        max_seq_length = 1024,
        dtype = torch.float16,
        load_in_4bit = True,
        load_in_8bit = False,
        full_finetuning = False,
    )

    print(
        "Model size",
        base_model.get_memory_footprint() / (1024**3),
        "GiB",
    )

    return base_model, base_tokenizer

In [14]:
# Dung lượng model được in trong load_base_model() trên Modal.
# Không truy cập base_model trực tiếp từ Colab.

In [15]:
def add_lora(base_model):
    from unsloth import FastLanguageModel

    # -----------------------------
    # 4. ADD LORA
    # -----------------------------
    model = FastLanguageModel.get_peft_model(
        base_model,
        r = 16,
        lora_alpha = 32,
        lora_dropout = 0,
        bias = "none",
        target_modules = [
            "q_proj","k_proj","v_proj","o_proj",
            "gate_proj","up_proj","down_proj",
        ],
        use_gradient_checkpointing = "unsloth"
    )

    return model

In [16]:
def prepare_tokenizer(base_tokenizer):
    from unsloth.chat_templates import get_chat_template

    tokenizer = get_chat_template(
        base_tokenizer,
        chat_template = "qwen3-instruct",
    )

    return tokenizer

In [17]:
# -----------------------------
# 1. LOAD DATASET
# 2. PROMPT + JSON TARGET
# -----------------------------

# Dùng tên riêng để không bị cell baseline ghi đè.
TRAIN_USER_TEMPLATE = """Bạn là hệ thống kiểm duyệt nội dung tiếng Việt.

Định nghĩa nhãn:
1. CLEAN
Câu không chứa ngôn từ quấy rối, không có từ ngữ thô tục, và không tấn công bất kỳ ai.
2. OFFENSIVE
Câu có chứa nội dung quấy rối hoặc từ ngữ thô tục, nhưng không nhắm vào một đối tượng cụ thể nào, để than phiền chung chung.
3. HATE
Câu chứa nội dung quấy rối, lăng mạ và nhắm trực tiếp vào một cá nhân hoặc một nhóm người cụ thể dựa trên các đặc điểm của họ (như đặc điểm cá nhân, tôn giáo, quốc tịch, chủng tộc, vùng miền...).
Lưu ý: Nếu có chửi thề nhưng không nhắm vào ai cụ thể, phải chọn OFFENSIVE. Chỉ chọn HATE khi có mục tiêu bị nhắm đến cụ thể.

Nhiệm vụ: phân loại câu dưới vào MỘT trong các nhãn CLEAN | OFFENSIVE | HATE
Câu:
\"\"\"{text}\"\"\"

Chỉ trả về JSON hợp lệ với schema:
{{
  "explanation": "Một lời giải thích ngắn gọn lý do phân loại như vậy (sử dụng tiếng Việt)",
  "evidence": ["Trích nguyên văn những từ ngữ, đoạn vi phạm (nếu có)"],
  "label": "CLEAN | OFFENSIVE | HATE"
}}"""


def prepare_dataset(tokenizer, user_template):
    from datasets import load_dataset
    import json

    # -----------------------------
    # 1. LOAD DATASET
    # 2. PROMPT + JSON TARGET
    # -----------------------------
    dataset = load_dataset("vominhmanh/vihsd-explainable")

    def formatting_prompts_func(examples):
        convos = []

        # Vì batched=True, examples là dict chứa list các giá trị.
        for text, label, explanation, evidence in zip(
            examples["text"],
            examples["label"],
            examples["explanation"],
            examples["evidence"],
        ):
            # 1. Tạo nội dung JSON mong muốn.
            target_json = {
                "explanation": explanation,
                "evidence": evidence,
                "label": label,
            }

            assistant_content = json.dumps(
                target_json,
                ensure_ascii=False,
            )

            # 2. Tạo cấu trúc hội thoại.
            conversation = [
                {
                    "role": "user",
                    "content": user_template.format(text=text),
                },
                {
                    "role": "assistant",
                    "content": assistant_content,
                },
            ]

            convos.append(conversation)

        # 3. Áp dụng template của tokenizer.
        texts = [
            tokenizer.apply_chat_template(
                convo,
                tokenize=False,
                add_generation_prompt=False,
            )
            for convo in convos
        ]

        return {"text": texts}

    dataset = dataset.map(
        formatting_prompts_func,
        batched=True,
    )

    return dataset


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import uuid
from transformers.trainer_callback import PrinterCallback

# Mỗi lần chạy có thư mục riêng, tránh ghi đè kết quả trước.
TRAIN_RUN_ID = "run-" + uuid.uuid4().hex[:12]


@app.function(
    image=image,
    gpu="H200",
    cpu=4,
    memory=32768,
    volumes={"/data": volume},
    secrets=[wandb_secret],
    timeout=6 * 60 * 60,
)
def train_vihsd_modal(user_template, run_id):
    import os

    # =============================
    # 1. LOAD MODEL + LORA + DATASET
    # =============================
    # Các hàm đã định nghĩa ở những cell trước.
    # Tất cả được gọi bên trong job H200.
    base_model, base_tokenizer = load_base_model()
    model = add_lora(base_model)

    tokenizer = prepare_tokenizer(base_tokenizer)
    dataset = prepare_dataset(tokenizer, user_template)

    # Import TRL sau khi Unsloth đã được import trong load_base_model().
    import inspect
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback

    # =============================
    # 2. SET PATH
    # =============================
    LOCAL_BASE = os.path.join("/data/qwen3_vihsd", run_id)

    LOCAL_CKPT = os.path.join(LOCAL_BASE, "checkpoints")
    LOCAL_LORA = os.path.join(LOCAL_BASE, "lora_model")

    os.makedirs(LOCAL_CKPT, exist_ok=True)
    os.makedirs(LOCAL_LORA, exist_ok=True)

    print("📂 Run:", run_id)
    print("📂 Checkpoints:", LOCAL_CKPT)
    print("📂 Adapter:", LOCAL_LORA)

    os.environ["WANDB_MODE"] = "online"
    os.environ["WANDB_DIR"] = LOCAL_BASE
    os.environ["WANDB_CONSOLE"] = "off"
    os.environ["WANDB_SILENT"] = "true"

    assert os.environ.get("WANDB_API_KEY"), (
        "Chưa truyền WANDB_API_KEY vào job Modal."
    )

    import wandb

    wandb_run = wandb.init(
        project="qwen3-vihsd",
        name=run_id,
        dir=LOCAL_BASE,
        mode="online",
    )

    # Persist checkpoint sau mỗi lần Trainer lưu.
    class CommitVolumeCallback(TrainerCallback):
        def on_save(self, args, state, control, **kwargs):
            volume.commit()
            return control

    # =============================
    # 3. TƯƠNG THÍCH PHIÊN BẢN TRL
    # =============================
    trainer_params = inspect.signature(
        SFTTrainer.__init__
    ).parameters

    config_params = inspect.signature(
        SFTConfig
    ).parameters

    extra_config = {}
    extra_trainer = {}

    # Các phiên bản TRL đặt hai tham số này ở vị trí khác nhau.
    for key, value in {
        "dataset_text_field": "text",
        "packing": True,
    }.items():
        if key in config_params:
            extra_config[key] = value
        elif key in trainer_params:
            extra_trainer[key] = value
        else:
            raise RuntimeError(
                f"Phiên bản TRL không có tham số bắt buộc: {key}"
            )

    if "max_length" in config_params:
        extra_config["max_length"] = 1024
    elif "max_seq_length" in config_params:
        extra_config["max_seq_length"] = 1024
    elif "max_seq_length" in trainer_params:
        extra_trainer["max_seq_length"] = 1024
    else:
        raise RuntimeError(
            "Không tìm thấy tham số sequence length trong TRL."
        )

    if "processing_class" in trainer_params:
        extra_trainer["processing_class"] = tokenizer
    elif "tokenizer" in trainer_params:
        extra_trainer["tokenizer"] = tokenizer
    else:
        raise RuntimeError(
            "Không tìm thấy tokenizer/processing_class trong TRL."
        )

    # =============================
    # 4. TRAINER
    # =============================
    trainer = SFTTrainer(
        model = model,
        train_dataset = dataset["train"],
        eval_dataset = dataset["validation"],
        args = SFTConfig(
            output_dir = LOCAL_CKPT,

            # Giữ nguyên cấu hình train gốc.
            per_device_train_batch_size = 2,
            per_device_eval_batch_size = 2,
            gradient_accumulation_steps = 2,
            learning_rate = 2e-4,
            num_train_epochs = 1,
            logging_steps = 50,
            eval_steps = 300,
            eval_strategy = "steps",
            bf16 = False,
            fp16 = True,
            optim = "adamw_8bit",
            lr_scheduler_type = "cosine",
            warmup_steps = 25,
            report_to = "wandb",
            logging_first_step = True,
            disable_tqdm = True,
            dataloader_num_workers = 2,
            gradient_checkpointing = True,
            run_name = "qwen3-4b-vihsd-lora-run-1",

            # Bổ sung lưu checkpoint lên Volume.
            save_strategy = "steps",
            save_steps = 300,
            save_total_limit = 2,

            **extra_config,
        ),
        callbacks = [CommitVolumeCallback()],
        **extra_trainer,
    )

    # Không tự bật train_on_responses_only:
    # bản gốc chỉ import nhưng chưa gọi hàm này.

    # =============================
    # 5. TRAIN
    # =============================
    trainer.remove_callback(PrinterCallback)
    train_result = trainer.train()

    # =============================
    # 6. SAVE ADAPTER
    # =============================
    print("💾 Saving adapter to Modal Volume...")

    trainer.save_model(LOCAL_LORA)
    tokenizer.save_pretrained(LOCAL_LORA)

    wandb_url = wandb_run.url
    wandb_run.finish()  # Chờ đồng bộ phần log còn lại.

    # Bảo đảm dữ liệu được persist trước khi job kết thúc.
    volume.commit()

    print("✅ DONE! Adapter saved:", LOCAL_LORA)

    # Chỉ trả thông tin về Colab, không trả model GPU.
    return {
        "run_id": run_id,
        "local_base": LOCAL_BASE,
        "checkpoint_path": LOCAL_CKPT,
        "adapter_path": LOCAL_LORA,
        "global_step": train_result.global_step,
        "training_loss": float(train_result.training_loss),
        "wandb_url": wandb_url,
    }


# =============================
# 7. GỌI JOB TRAIN TỪ COLAB
# =============================
print("Bắt đầu train trên H200. Run:", TRAIN_RUN_ID)

with modal.enable_output():
    with app.run(client=modal_client):
        print("Job Modal:", app.get_dashboard_url())
        print("Theo dõi metric trong project qwen3-vihsd trên W&B.")

        train_result = train_vihsd_modal.remote(
            TRAIN_USER_TEMPLATE,
            TRAIN_RUN_ID,
        )
    print("W&B run:", train_result["wandb_url"])

# Giữ đường dẫn để dùng cho cell inference tiếp theo.
# Đây là đường dẫn trên Modal, không phải trên Colab.
LOCAL_BASE = train_result["local_base"]
LOCAL_CKPT = train_result["checkpoint_path"]
LOCAL_LORA = train_result["adapter_path"]

print("\n===== TRAIN RESULT =====")
print("Run:", train_result["run_id"])
print("Global step:", train_result["global_step"])
print("Training loss:", train_result["training_loss"])
print("Adapter trên Modal:", LOCAL_LORA)

Bắt đầu train trên H200. Run: run-fdfeaea1d367
✓ Initialized. View run at 
https://modal.com/apps/luluchi19/main/ap-IzmfMGeZEklYGVA16qZTTF
✓ Created objects.
├── ✓ Created Function check_modal_gpu.
├── ✓ Created Function infer_baseline_modal.
├── ✓ Created Function evaluate_baseline_modal.
└── ✓ Created Function train_vihsd_modal.
Job Modal: https://modal.com/id/ap-IzmfMGeZEklYGVA16qZTTF
Theo dõi metric trong project qwen3-vihsd trên W&B.

== CUDA ==

CUDA Version 12.8.1

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.

W1030 16:34:37.164000 2 site-packages/torch/utils/_pytre

In [20]:
import os
import sys
import subprocess

# Lưu mỗi run trong một thư mục riêng trên Drive.
DRIVE_BASE = os.path.join(
    "/content/drive/MyDrive/qwen3_vihsd",
    train_result["run_id"],
)
DRIVE_LORA = os.path.join(DRIVE_BASE, "lora_model")

assert os.path.isdir("/content/drive/MyDrive"), (
    "Hãy chạy cell drive.mount trước."
)

os.makedirs(DRIVE_BASE, exist_ok=True)

# /data là điểm mount trong máy Modal.
# Đường dẫn bên trong Volume không bao gồm tiền tố /data.
adapter_volume_path = train_result["adapter_path"].removeprefix("/data")

print("🚀 Copying adapter from Modal to Google Drive...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "modal",
        "volume",
        "get",
        "qwen3-vihsd-data",
        adapter_volume_path,
        DRIVE_LORA,
    ],
    check=True,
)

print("✅ Adapter trên Modal:", train_result["adapter_path"])
print("✅ Bản sao trên Google Drive:", DRIVE_LORA)

🚀 Copying adapter from Modal to Google Drive...
✅ Adapter trên Modal: /data/qwen3_vihsd/run-fdfeaea1d367/lora_model
✅ Bản sao trên Google Drive: /content/drive/MyDrive/qwen3_vihsd/run-fdfeaea1d367/lora_model


# Infer sử dụng model fine-tuned

In [21]:
# Dùng đúng adapter được job train trước trả về.
assert (
    "train_result" in globals()
    and "adapter_path" in train_result
), "Chưa có đường dẫn adapter. Kiểm tra kết quả cell train trước."

LOCAL_BASE = train_result["local_base"]
LOCAL_CKPT = train_result["checkpoint_path"]
LOCAL_LORA = train_result["adapter_path"]

print("Run đã train:", train_result["run_id"])
print("Adapter trên Modal:", LOCAL_LORA)
print("Inference sẽ đọc trực tiếp adapter này, không cần copy từ Drive.")

Run đã train: run-fdfeaea1d367
Adapter trên Modal: /data/qwen3_vihsd/run-fdfeaea1d367/lora_model
Inference sẽ đọc trực tiếp adapter này, không cần copy từ Drive.


In [22]:
# ---------- CONFIG ----------
MODEL_NAME = LOCAL_LORA
BATCH_SIZE = 32

# Giữ nguyên prompt của cell đánh giá gốc.
FINETUNED_USER_TEMPLATE = """Bạn là hệ thống kiểm duyệt nội dung tiếng Việt.

Định nghĩa nhãn:
1. CLEAN
Câu không chứa ngôn từ quấy rối, không có từ ngữ thô tục, và không tấn công bất kỳ ai.
2. OFFENSIVE
Câu có chứa nội dung quấy rối hoặc từ ngữ thô tục, nhưng không nhắm vào một đối tượng cụ thể nào, để than phiền chung chung.
3. HATE
Câu chứa nội dung quấy rối, lăng mạ và nhắm trực tiếp vào một cá nhân hoặc một nhóm người cụ thể dựa trên các đặc điểm của họ (như đặc điểm cá nhân, tôn giáo, quốc tịch, chủng tộc, vùng miền...).
Lưu ý: Nếu có chửi thề nhưng không nhắm vào ai cụ thể, phải chọn OFFENSIVE. Chỉ chọn HATE khi có mục tiêu bị nhắm đến cụ thể.

Nhiệm vụ: phân loại câu dưới vào MỘT trong các nhãn CLEAN | OFFENSIVE | HATE
Câu:
\"\"\"{text}\"\"\"

Chỉ trả về JSON hợp lệ với schema:
{{
  "explanation": "Một lời giải thích ngắn gọn lý do phân loại như vậy (sử dụng tiếng Việt)",
  "evidence": ["Trích nguyên văn những từ ngữ, đoạn vi phạm (nếu có)"],
  "label": "CLEAN | OFFENSIVE | HATE"
}}"""


# ---------- CHẠY TRÊN MODAL ----------
@app.function(
    image=image,
    gpu="H200",
    cpu=4,
    memory=32768,
    volumes={"/data": volume},
    timeout=6 * 60 * 60,
)
def evaluate_finetuned_modal(adapter_path, user_template, batch_size):
    from unsloth import FastLanguageModel
    import os
    import json
    import torch
    from datasets import load_dataset
    from sklearn.metrics import f1_score
    from tqdm import tqdm

    DEVICE = "cuda"

    assert torch.cuda.is_available(), "Máy Modal chưa có CUDA."

    gpu = torch.cuda.get_device_properties(0)
    print("GPU phía Modal:", gpu.name)
    print("VRAM:", round(gpu.total_memory / 1024**3, 1), "GiB")

    # Kiểm tra adapter ngay trên máy Modal.
    assert os.path.isfile(
        os.path.join(adapter_path, "adapter_config.json")
    ), f"Không tìm thấy adapter_config.json tại {adapter_path}"

    # ---------- PARSE (NO FALLBACK) ----------
    def extract_label(text):
        obj = json.loads(text)
        return obj["label"].strip().upper()

    # ---------- LOAD MODEL FINE-TUNED ----------
    print(f"Loading fine-tuned model: {adapter_path}")

    # Load trực tiếp thư mục adapter đã lưu.
    # Unsloth sẽ đọc cấu hình để tải base model và adapter.
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=adapter_path,
        max_seq_length=1024,
        dtype=torch.float16,
        load_in_4bit=True,
    )

    FastLanguageModel.for_inference(model)

    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.eos_token

    # ---------- LOAD DATA ----------
    ds = load_dataset("vominhmanh/vihsd-explainable")["test"]

    y_true, y_pred = [], []

    print(
        f"Running inference on {len(ds)} samples "
        f"(BS={batch_size})..."
    )

    # ---------- INFERENCE ----------
    for i in tqdm(
        range(0, len(ds), batch_size),
        mininterval=60,
        miniters=1,
    ):
        batch = ds[i:i + batch_size]

        texts = batch["text"]
        labels = batch["label"] if "label" in batch else [
            ["CLEAN", "OFFENSIVE", "HATE"][lid]
            for lid in batch["label_id"]
        ]

        prompts = [
            tokenizer.apply_chat_template(
                [
                    {
                        "role": "user",
                        "content": user_template.format(text=t),
                    }
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            for t in texts
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.7,
            )

        gen_ids = outputs[:, inputs["input_ids"].shape[1]:]
        decoded = tokenizer.batch_decode(
            gen_ids,
            skip_special_tokens=True,
        )

        for t, raw, gold in zip(texts, decoded, labels):
            try:
                pred = extract_label(raw)
            except:
                pred = "ERROR"

            y_true.append(gold)
            y_pred.append(pred)

    # ---------- F1: GIỮ CÁCH TÍNH GỐC ----------
    valid = [
        (t, p)
        for t, p in zip(y_true, y_pred)
        if t is not None
    ]
    y_t, y_p = zip(*valid)

    f1 = f1_score(y_t, y_p, average="macro")

    return {
        "adapter_path": adapter_path,
        "samples": len(y_t),
        "f1": float(f1),
        "y_true": y_true,
        "y_pred": y_pred,
        "error_count": y_pred.count("ERROR"),
    }


# ---------- GỌI H200 TỪ COLAB ----------
with modal.enable_output():
    with app.run(client=modal_client):
        print("Đang đánh giá fine-tuned. Theo dõi tại:", app.get_dashboard_url())

        finetuned_result = evaluate_finetuned_modal.remote(
            MODEL_NAME,
            FINETUNED_USER_TEMPLATE,
            BATCH_SIZE,
        )

# ---------- KẾT QUẢ TẠI COLAB ----------
y_true = finetuned_result["y_true"]
y_pred = finetuned_result["y_pred"]
f1 = finetuned_result["f1"]

print("\n===== FINE-TUNED RESULT =====")
print("Adapter:", finetuned_result["adapter_path"])
print(f"Samples: {finetuned_result['samples']}")
print(f"Macro F1: {f1:.4f}")
print("Số mẫu trả về ERROR:", finetuned_result["error_count"])

# So sánh nếu kết quả baseline vẫn còn trong Colab.
if "baseline_result" in globals():
    print("\n===== BASELINE vs FINE-TUNED =====")
    print(f"Baseline Macro F1:   {baseline_result['f1']:.4f}")
    print(f"Fine-tuned Macro F1: {finetuned_result['f1']:.4f}")
    print(
        "Chênh lệch:",
        f"{finetuned_result['f1'] - baseline_result['f1']:+.4f}",
    )

✓ Initialized. View run at 
https://modal.com/apps/luluchi19/main/ap-yhzOWUJ8gfEuiwQELhuufm
✓ Created objects.
├── ✓ Created Function check_modal_gpu.
├── ✓ Created Function infer_baseline_modal.
├── ✓ Created Function evaluate_baseline_modal.
├── ✓ Created Function train_vihsd_modal.
└── ✓ Created Function evaluate_finetuned_modal.
Đang đánh giá fine-tuned. Theo dõi tại: https://modal.com/id/ap-yhzOWUJ8gfEuiwQELhuufm

== CUDA ==

CUDA Version 12.8.1

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W103

In [23]:
import os
import json

assert os.path.isdir("/content/drive/MyDrive"), (
    "Hãy mount Google Drive trước khi lưu kết quả."
)

RESULT_DIR = os.path.join(
    "/content/drive/MyDrive/qwen3_vihsd",
    train_result["run_id"],
)
os.makedirs(RESULT_DIR, exist_ok=True)

RESULT_PATH = os.path.join(RESULT_DIR, "finetuned_evaluation.json")

with open(RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        finetuned_result,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("✅ Đã lưu kết quả:", RESULT_PATH)

✅ Đã lưu kết quả: /content/drive/MyDrive/qwen3_vihsd/run-fdfeaea1d367/finetuned_evaluation.json
